In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Annotation Pipeline

## Giai đoạn 1: Khởi tạo tri thức (Teacher Generation)
Sử dụng Gemini 1.5 Pro hoặc GPT-4o làm Annotator chính.

Nạp bộ quy tắc nhãn (5 nhãn trên) và kỹ thuật Reasoning Scaffolding vào System Prompt.

Sản phẩm: Một bộ Dataset thô gồm Input -> Thought -> Label.

## phase 1.1 : first check 

In [28]:
%%writefile prompt_phase_1.txt

---

**Role:** You are a linguistics expert specializing in Vietnamese social media culture and a professional Content Moderator. You are highly skilled at decoding teencode, understanding Gen Z language, and detecting sarcasm with great sensitivity.

---

**Task:** Analyze user content based on the provided **Text** and **Emotion**. You must perform step-by-step reasoning to classify the content into **one of five target labels**.

---

### **1. Label System (Final Labels):**

* **Constructive/Clean:** Normal information, positive feedback, or harmless personal emotional sharing.
* **Implicit Toxicity:** Sarcasm, irony, or seemingly positive wording used to insult or attack.
* **Explicit Hostility:** Direct insults, profanity, offensive language, or explicit threats.
* **Identity-Based Hate:** Discrimination based on region, gender, religion, or appearance (body shaming).
* **Ambiguous/Noise:** Too short, only emojis, heavily misspelled, or insufficient information to interpret.

---

### **2. Reasoning Process (Reasoning Scaffolding):**

You are required to analyze through the following steps:

* **Step 1: Semantic & Slang Decoding**
  Decode slang, teencode, and the literal/figurative meanings of the sentence.

* **Step 2: Contextual Conflict**
  Compare the textual content with the associated emotion.
  *(Example: Emotion is "Angry" but the text is praise → likely sarcasm.)*

* **Step 3: Target Identification**
  Identify who the statement is directed at (self, another individual, or a specific group).

* **Step 4: Thought Trace**
  Synthesize logic from the above steps to form a reasoning chain.

---

### **3. Output Format (JSON):**

Return **only** the result in the following JSON format:

```json
{
  "reasoning_scaffolding": {
    "semantic_decoding": "Decode wording and sentence structure...",
    "slang_interpretation": "Explain detected slang or teencode...",
    "contextual_conflict": "Analyze alignment between text and emotion...",
    "target": "Identify the target..."
  },
  "thought_trace": "Final logical reasoning before assigning label...",
  "final_label": "Label name",
  "confidence_score": "Confidence score (0.0 - 1.0)"
}
```

---

### **INPUT SAMPLES**

Below are some examples (use this structure when calling the API):

**Sample 1:**

```json
{
  "text": "khum bít bao giờ mới đc đi chơi vs ny nhỉ, bùn ghê",
  "emotion": "sad"
}
```

**Sample 2:**

```json
{
  "text": "thằng bắc kỳ này lại bắt đầu gáy rồi đấy",
  "emotion": "angry"
}
```

---

### **EXPECTED MODEL RESPONSE (EXAMPLE)**
Return in vietnamese. 
```json
{
  "reasoning_scaffolding": {
    "semantic_decoding": "Cụm từ 'giỏi quá' và 'cả họ tự hào' mang nghĩa tích cực. Tuy nhiên, 'vcl' là từ cảm thán mạnh thường dùng trong bối cảnh tiêu cực hoặc suồng sã.",
    "slang_interpretation": "vcl = vãi cả lồn (từ cảm thán mạnh).",
    "contextual_conflict": "Emotion 'smirk' (cười nhếch mép) mâu thuẫn với lời khen 'giỏi quá'. Điều này xác nhận đây là lời mỉa mai (Sarcasm).",
    "target": "Cá nhân người đối diện và gia đình của họ."
  },
  "thought_trace": "Mặc dù câu chữ mang vẻ khen ngợi nhưng kết hợp với cảm xúc nhếch mép và từ cảm thán thô tục, mục đích là để hạ nhục đối phương thông qua hình thức châm biếm.",
  "final_label": "Implicit Toxicity",
  "confidence_score": 0.95,
  "suggested_action": "Review"
}
```

Overwriting prompt_phase_1.txt
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [9]:
import os

os.environ['api_key'] = "ssk-xxx"

In [ ]:
%%writefile phase_1.py
import asyncio
import json
import logging
import os
import argparse
from typing import List, Dict, Any
import pandas as pd
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

api_key = ""

# --- CẤU HÌNH LOGGING ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("annotation.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)

class TeacherAnnotator:
    def __init__(self, prompt_path: str, model: str = "gpt-4o"):
        self.client = AsyncOpenAI(api_key=api_key)
        self.model = model
        self.prompt_path = prompt_path
        self.system_prompt = self._load_system_prompt()

    def _load_system_prompt(self) -> str:
        """Đọc prompt từ file .txt"""
        if not os.path.exists(self.prompt_path):
            raise FileNotFoundError(f"Không tìm thấy file prompt tại: {self.prompt_path}")
        with open(self.prompt_path, 'r', encoding='utf-8') as f:
            return f.read().strip()

    async def annotate_single(self, text: str, emotion: str, semaphore: asyncio.Semaphore) -> Dict:
        """Xử lý một dòng dữ liệu đơn lẻ"""
        async with semaphore:
            try:
                user_content = f"Input Text: '{text}'\nInput Emotion: '{emotion}'"
                
                response = await self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": self.system_prompt},
                        {"role": "user", "content": user_content}
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.2
                )
                
                raw_res = response.choices[0].message.content
                result = json.loads(raw_res)
                
                # Làm phẳng cấu trúc JSON để lưu CSV dễ dàng hơn
                flat_res = {
                    "input_text": text,
                    "input_emotion": emotion,
                    "semantic_decoding": result.get("reasoning_scaffolding", {}).get("semantic_decoding"),
                    "slang_interpretation": result.get("reasoning_scaffolding", {}).get("slang_interpretation"),
                    "contextual_conflict": result.get("reasoning_scaffolding", {}).get("contextual_conflict"),
                    "target": result.get("reasoning_scaffolding", {}).get("target"),
                    "thought_trace": result.get("thought_trace"),
                    "final_label": result.get("final_label"),
                    "confidence_score": result.get("confidence_score")
                }
                return flat_res

            except Exception as e:
                logging.error(f"Lỗi khi xử lý: {text[:30]}... | Lỗi: {e}")
                return {"input_text": text, "input_emotion": emotion, "error": str(e)}

    async def annotate_batch(self, data: List[Dict[str, str]], max_concurrent: int):
        """Xử lý theo từng cụm nhỏ để tránh Rate Limit TPM"""
        semaphore = asyncio.Semaphore(max_concurrent)
        all_results = []
        
        # Chia dữ liệu thành các nhóm nhỏ (ví dụ mỗi nhóm 20 records)
        chunk_size = max_concurrent * 2 
        for i in range(0, len(data), chunk_size):
            chunk = data[i:i + chunk_size]
            tasks = [self.annotate_single(item['text'], item['emotion'], semaphore) for item in chunk]
            
            chunk_results = await tqdm.gather(*tasks, desc=f"Đang xử lý cụm {i//chunk_size + 1}", leave=False)
            all_results.extend(chunk_results)
            
            # Nghỉ 5 giây giữa các cụm để reset Token Per Minute (TPM)
            logging.info("Đang nghỉ 5s để tránh Rate Limit...")
            await asyncio.sleep(5)
            
        return all_results

def parse_args():
    parser = argparse.ArgumentParser(description="Teacher Knowledge Generation Pipeline")
    parser.add_argument('--input', type=str, required=True, help="Path tới file CSV đầu vào")
    parser.add_argument('--output', type=str, required=True, help="Path lưu file kết quả")
    parser.add_argument('--prompt', type=str, default='system_prompt.txt', help="Path tới file prompt hệ thống")
    parser.add_argument('--model', type=str, default='gpt-4o', help="Model OpenAI (gpt-4o, gpt-4-turbo)")
    parser.add_argument('--batch_size', type=int, default=10, help="Số lượng request gửi song song (Max concurrent)")
    parser.add_argument('--text_col', type=str, default='text', help="Tên cột chứa nội dung text")
    parser.add_argument('--emotion_col', type=str, default='emotion', help="Tên cột chứa cảm xúc")
    parser.add_argument('--size_pct', type=float, help="Lấy khoảng bao nhiêu dataset")
    parser.add_argument('--test', action='store_true', help="Chạy chế độ test với 5 dòng đầu tiên")
    return parser.parse_args()

async def test_logic(args):
    """Hàm chạy test nhanh với các mẫu dữ liệu thực tế mạng xã hội để kiểm tra LLM Reasoning"""
    logging.info("--- ĐANG CHẠY TEST LOGIC NỘI BỘ (LLM REASONING) ---")
    
    # Khởi tạo annotator dành riêng cho test
    annotator = TeacherAnnotator(
        prompt_path=args.prompt, 
        model=args.model
    )

    # Dữ liệu test bao gồm text và emotion giả định
    test_samples = [
        {"text": "Hôm nay t đi học trễ vcl 😂😂😂, đm thầy giáo gắt quá !!!", "emotion": "angry"},
        {"text": "Khum bít bao giờ mới đc đi chơi vs ny nhỉ ❤️✨", "emotion": "sad"},
        {"text": "mng ơi mik mới mua cái đt mới xịn xò lắm lun 📱📱📱", "emotion": "happy"},
        {"text": "clgt sao m lại làm thế vs t 😡", "emotion": "angry"},
        {"text": "chằm zn lun á mng ơi ét ô ét 🆘", "emotion": "fear"}
    ]

    print("\n" + "="*30 + " TEST LLM REASONING RESULTS " + "="*30)
    
    # Chạy xử lý thông qua batch (với max_concurrent nhỏ cho test)
    results = await annotator.annotate_batch(test_samples, max_concurrent=2)

    for res in results:
        if "error" in res:
            print(f"[!] Lỗi: {res['error']}")
            continue
            
        print(f"\n[-] Input:  {res['input_text']}")
        print(f"[-] Emotion: {res['input_emotion']}")
        print(f"[>] Thought Trace: {res['thought_trace']}")
        print(f"[+] Final Label: {res['final_label']} (Score: {res['confidence_score']})")
        print("-" * 80)
        
    result_df = pd.DataFrame(results)
    result_df.to_csv(args.output, index=False, encoding='utf-8-sig')
    logging.info(f"Hoàn tất! Kết quả đã được lưu tại: {args.output}")

async def run_pipeline(args):
    # Khởi tạo Annotator
    annotator = TeacherAnnotator(
        prompt_path=args.prompt, 
        model=args.model
    )

    logging.info(f"Đang đọc dữ liệu từ {args.input}...")
    try:
        df = pd.read_csv(args.input)
    except Exception as e:
        logging.error(f"Không thể đọc file input: {e}")
        return
    
    # Chế độ lấy mẫu nếu cần (không ghi đè head(5) của test_logic)
    process_df = df.head(10) if args.test else df
    tmp = int(args.size_pct * len(process_df))
    process_df = process_df.iloc[:tmp]

    records = process_df.rename(
        columns={args.text_col: 'text', args.emotion_col: 'emotion'}
    ).to_dict('records')
    
    all_results = await annotator.annotate_batch(records, max_concurrent=args.batch_size)

    result_df = pd.DataFrame(all_results)
    result_df.to_csv(args.output, index=False, encoding='utf-8-sig')
    logging.info(f"Hoàn tất! Kết quả đã được lưu tại: {args.output}")

if __name__ == "__main__":
    args = parse_args()
    
    # Nếu có flag --test, chỉ chạy test_logic rồi thoát
    if args.test:
        asyncio.run(test_logic(args))
    else:
        # Nếu không, chạy pipeline bình thường
        asyncio.run(run_pipeline(args))

Overwriting phase_1.py


In [30]:
!python phase_1.py \
    --input /kaggle/input/datasets/quyenuit24/hate-speech-detection/test_nor_811.xlsx \
    --output test_phase_1_openai4o.csv \
    --prompt /kaggle/working/prompt_phase_1.txt \
    --batch_size 5 \
    --text_col "Sentence" \
    --emotion_col "Emotion" \
    --test

2026-04-22 08:22:32,118 - INFO - --- ĐANG CHẠY TEST LOGIC NỘI BỘ (LLM REASONING) ---

============================== TEST LLM REASONING RESULTS ==============================
Đang xử lý batch:  80%|█████████████████████▌     | 4/5 [00:07<00:01,  1.52s/it]2026-04-22 08:22:41,237 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
                                                                                
[-] Input:  Hôm nay t đi học trễ vcl 😂😂😂, đm thầy giáo gắt quá !!!
[-] Emotion: angry
[>] Thought Trace: Người nói sử dụng ngôn ngữ thô tục để thể hiện sự bực bội với thầy giáo. Mặc dù có biểu tượng cảm xúc cười, nhưng từ ngữ và cảm xúc 'angry' cho thấy sự không hài lòng rõ ràng.
[+] Final Label: Explicit Hostility (Score: 0.9)
--------------------------------------------------------------------------------

[-] Input:  Khum bít bao giờ mới đc đi chơi vs ny nhỉ ❤️✨
[-] Emotion: sad
[>] Thought Trace: Mặc dù cảm xúc là 'buồn', nhưng việc sử dụng 

In [48]:
!python phase_1.py \
    --input /kaggle/input/datasets/quyenuit24/cleaned-hate-speech-dataset/processed_train.csv \
    --output test_phase_1.csv \
    --prompt /kaggle/working/prompt_phase_1.txt \
    --batch_size 2 \
    --text_col "Sentence" \
    --emotion_col "Emotion" \
    --size_pct 0.5

2026-04-22 08:39:40,312 - INFO - Đang đọc dữ liệu từ /kaggle/input/datasets/quyenuit24/cleaned-hate-speech-dataset/processed_train.csv...
Đang xử lý cụm 1:  75%|████████████████████▎      | 3/4 [00:07<00:02,  2.37s/it]2026-04-22 08:39:48,792 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 08:39:48,795 - INFO - Đang nghỉ 5s để tránh Rate Limit...            
Đang xử lý cụm 2:  25%|██████▊                    | 1/4 [00:02<00:08,  2.89s/it]2026-04-22 08:39:56,747 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 08:39:58,685 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Đang xử lý cụm 2:  75%|████████████████████▎      | 3/4 [00:04<00:01,  1.49s/it]2026-04-22 08:39:58,747 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-22 08:39:58,752 - INFO - Đang nghỉ 5s để tránh Rate Limit...            
Đang xử 

## Giai đoạn 2: Kiểm chứng chéo (AI-Cross-Check)
Dùng một Model khác (như Claude 3.5 hoặc Qwen-Max) đóng vai Kiểm soát viên.

Kiểm tra xem phần Thought của Teacher ở Giai đoạn 1 có bị "ảo giác" (hallucination) hay không.

Luật: Chỉ giữ lại những mẫu dữ liệu mà cả 2 Model đồng thuận về Label và logic suy luận không mâu thuẫn.

## test LLMs providers

In [ ]:
!pip install anthropic
!pip install google-generativeai

In [ ]:
import os
import os
os.environ["ANTHROPIC_API_KEY"] = ""

In [11]:
os.environ['OPENAI_API_KEY'] = 'sk-proj-x'

In [15]:
from anthropic import Anthropic
from openai import OpenAI
import os

client = Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))

# Liệt kê tất cả các model khả dụng
models = client.models.list()

for model in models:
    print(f"ID: {model.id}")
    print(f"Name: {model.display_name}")
    print(f"Context Window: {model.max_input_tokens}")
    print("-" * 20)

ID: claude-opus-4-7
Name: Claude Opus 4.7
Context Window: 1000000
--------------------
ID: claude-sonnet-4-6
Name: Claude Sonnet 4.6
Context Window: 1000000
--------------------
ID: claude-opus-4-6
Name: Claude Opus 4.6
Context Window: 1000000
--------------------
ID: claude-opus-4-5-20251101
Name: Claude Opus 4.5
Context Window: 200000
--------------------
ID: claude-haiku-4-5-20251001
Name: Claude Haiku 4.5
Context Window: 200000
--------------------
ID: claude-sonnet-4-5-20250929
Name: Claude Sonnet 4.5
Context Window: 1000000
--------------------
ID: claude-opus-4-1-20250805
Name: Claude Opus 4.1
Context Window: 200000
--------------------


In [19]:
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

# Liệt kê tất cả các model khả dụng
models = client.models.list()

# Sắp xếp theo ID để dễ theo dõi (tùy chọn)
sorted_models = sorted(models.data, key=lambda x: x.id)

for model in sorted_models[:20]:
    print(f"ID: {model.id}")
    print(f"Created: {model.created}")
    print(f"Owned By: {model.owned_by}")
    print("-" * 20)

ID: babbage-002
Created: 1692634615
Owned By: system
--------------------
ID: chatgpt-image-latest
Created: 1765925279
Owned By: system
--------------------
ID: dall-e-2
Created: 1698798177
Owned By: system
--------------------
ID: dall-e-3
Created: 1698785189
Owned By: system
--------------------
ID: davinci-002
Created: 1692634301
Owned By: system
--------------------
ID: gpt-3.5-turbo
Created: 1677610602
Owned By: openai
--------------------
ID: gpt-3.5-turbo-0125
Created: 1706048358
Owned By: system
--------------------
ID: gpt-3.5-turbo-1106
Created: 1698959748
Owned By: system
--------------------
ID: gpt-3.5-turbo-16k
Created: 1683758102
Owned By: openai-internal
--------------------
ID: gpt-3.5-turbo-instruct
Created: 1692901427
Owned By: system
--------------------
ID: gpt-3.5-turbo-instruct-0914
Created: 1694122472
Owned By: system
--------------------
ID: gpt-4
Created: 1687882411
Owned By: openai
--------------------
ID: gpt-4-0613
Created: 1686588896
Owned By: openai
-----

# LLM provider